# Agents Laboratory - A Goal-Based Agent for Warehouse Navigation

**Shrihari Karthik (f20230868)** - *Artificial Intelligence: Agents Laboratory Exercise*

An autonomous warehouse vehicle must carry a package from the loading bay `S` to the dispatch
area `G`. Shelving units (`#`) cannot be crossed. The vehicle moves one grid square at a time,
Up / Down / Left / Right, and must follow a collision-free path.

This notebook implements the agent as a **goal-based agent**: it maintains an internal model of
the warehouse, an explicit goal, and a decision-making component (breadth-first search) that
plans a sequence of actions reaching that goal before any action is executed.

Run top to bottom. Section 6 contrasts a reflex agent, 7 is the test suite, 8 the
unreachable-goal case, and 9 the scaling study.

## 1. The environment: the warehouse grid

The map is stored exactly as given in the laboratory sheet and parsed into a 2-D grid.
Positions are `(row, col)` with row 0 at the top. Parsing is validated rather than assumed:
the map must be rectangular and must contain exactly one `S` and one `G`.

In [ ]:
from collections import deque
import heapq
import time

WAREHOUSE_MAP = """\
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################"""

OBSTACLE, FREE, START, GOAL = "#", ".", "S", "G"


def parse_map(text):
    """Parse the ASCII map into (grid, start, goal).

    grid  : list of lists of characters, grid[row][col]
    start : (row, col) of 'S'
    goal  : (row, col) of 'G'

    The parse is validated: the map must be rectangular and contain exactly one S and one G.
    Validating here means a malformed map fails loudly at load time instead of producing a
    confusing 'no path found' later on.
    """
    grid = [list(line) for line in text.strip("\n").split("\n")]

    widths = {len(r) for r in grid}
    if len(widths) != 1:
        raise ValueError(f"map is not rectangular; row widths seen: {sorted(widths)}")

    found = {START: [], GOAL: []}
    legal = {OBSTACLE, FREE, START, GOAL}
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch not in legal:
                raise ValueError(f"unexpected character {ch!r} at ({r}, {c})")
            if ch in found:
                found[ch].append((r, c))

    for symbol, name in ((START, "start"), (GOAL, "goal")):
        if len(found[symbol]) != 1:
            raise ValueError(f"expected exactly one {name} '{symbol}', found {len(found[symbol])}")

    return grid, found[START][0], found[GOAL][0]


grid, start, goal = parse_map(WAREHOUSE_MAP)
n_rows, n_cols = len(grid), len(grid[0])

print(WAREHOUSE_MAP)
print(f"\ngrid size : {n_rows} rows x {n_cols} columns = {n_rows * n_cols} cells")
print(f"start S   : {start}")
print(f"goal  G   : {goal}")
free_cells = sum(ch != OBSTACLE for row in grid for ch in row)
print(f"free cells: {free_cells}  (obstacles: {n_rows * n_cols - free_cells})")

## 2. Available actions and collision checking

Four actions, each moving the vehicle by one grid square. A move is legal only if the
destination is **inside the grid** and **is not an obstacle** - these two conditions together
are the collision check, and every part of the program that generates a move goes through
`is_free()`, so there is exactly one place where collision logic lives.

In [ ]:
# action name -> (row change, column change). Row 0 is the top row, so "Up" decreases row.
ACTIONS = {
    "Up":    (-1, 0),
    "Down":  (+1, 0),
    "Left":  (0, -1),
    "Right": (0, +1),
}


def in_bounds(grid, pos):
    """True if pos lies inside the grid."""
    r, c = pos
    return 0 <= r < len(grid) and 0 <= c < len(grid[0])


def is_free(grid, pos):
    """Collision check: True if the vehicle may occupy pos (inside the grid, not shelving)."""
    return in_bounds(grid, pos) and grid[pos[0]][pos[1]] != OBSTACLE


def successors(grid, pos):
    """Yield (action_name, next_pos) for every legal action from pos.

    This is the transition model. Illegal moves - off the grid, or into a shelving unit -
    are filtered out here, so no collision can ever enter the search tree.
    """
    r, c = pos
    for action, (dr, dc) in ACTIONS.items():
        nxt = (r + dr, c + dc)
        if is_free(grid, nxt):
            yield action, nxt


# quick sanity check of the collision test on known cells
print("start is free      :", is_free(grid, start))
print("goal is free       :", is_free(grid, goal))
print("corner (0,0) is free:", is_free(grid, (0, 0)), "(it is a wall)")
print("outside (-1,0) free :", is_free(grid, (-1, 0)), "(off the grid)")
print("\nlegal moves from the start", start, ":")
for action, nxt in successors(grid, start):
    print(f"   {action:<5} -> {nxt}")

## 3. The decision-making component: breadth-first search

Every action costs the same (one grid square), so **breadth-first search** finds a shortest
path: it expands states in order of increasing depth, therefore the first time it reaches `G`
it has reached it by a minimum number of moves. It is also complete - on a finite grid it
either finds a path or proves that none exists.

The `visited` set is what makes this tractable: the grid graph has cycles (the vehicle can
move right then left forever), so without it the search would not terminate in practice.
Marking a state when it is *enqueued* rather than when it is dequeued prevents the same cell
being pushed onto the frontier several times.

In [ ]:
def reconstruct(parents, start, goal):
    """Walk the parent pointers backwards from goal to start and return the forward path.

    parents maps state -> (previous_state, action_taken). Returns (path, actions).
    """
    path, actions = [goal], []
    node = goal
    while node != start:
        node, action = parents[node]
        path.append(node)
        actions.append(action)
    path.reverse()
    actions.reverse()
    return path, actions


def bfs(grid, start, goal):
    """Breadth-first search from start to goal.

    Returns a result dict with the path, the action sequence and search statistics.
    result['path'] is None when the goal is unreachable.
    """
    t0 = time.perf_counter()

    if not (is_free(grid, start) and is_free(grid, goal)):
        raise ValueError("start or goal is not a free cell")

    frontier = deque([start])          # FIFO queue -> explores by increasing depth
    visited = {start}                  # states already enqueued; prevents cycles and re-work
    parents = {}                       # state -> (previous state, action used to get here)
    expanded = 0                       # states removed from the frontier and expanded
    max_frontier = 1

    while frontier:
        current = frontier.popleft()
        expanded += 1

        if current == goal:            # goal test on expansion
            path, actions = reconstruct(parents, start, goal)
            return {"path": path, "actions": actions, "cost": len(actions),
                    "expanded": expanded, "max_frontier": max_frontier,
                    "visited": len(visited), "seconds": time.perf_counter() - t0}

        for action, nxt in successors(grid, current):
            if nxt not in visited:
                visited.add(nxt)
                parents[nxt] = (current, action)
                frontier.append(nxt)
        max_frontier = max(max_frontier, len(frontier))

    # frontier exhausted without reaching the goal: no path exists
    return {"path": None, "actions": None, "cost": None,
            "expanded": expanded, "max_frontier": max_frontier,
            "visited": len(visited), "seconds": time.perf_counter() - t0}

### A* with a Manhattan heuristic (for comparison)

Not required by the task, but it is the natural answer to the *Think About It* question about a
warehouse twice the size, and it lets the two algorithms be compared on the same map. The
Manhattan distance is admissible here - it never overestimates, because each move changes the
distance by at most one and diagonal moves are not allowed - so A* also returns a shortest path.

In [ ]:
def manhattan(a, b):
    """Admissible, consistent heuristic for 4-connected movement with unit cost."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(grid, start, goal):
    """A* search with the Manhattan heuristic. Same return format as bfs()."""
    t0 = time.perf_counter()
    counter = 0                                   # tie-breaker keeps the heap comparison total
    frontier = [(manhattan(start, goal), 0, counter, start)]
    best_g = {start: 0}
    parents = {}
    closed = set()
    expanded = 0
    max_frontier = 1

    while frontier:
        _, g, _, current = heapq.heappop(frontier)
        if current in closed:
            continue
        closed.add(current)
        expanded += 1

        if current == goal:
            path, actions = reconstruct(parents, start, goal)
            return {"path": path, "actions": actions, "cost": len(actions),
                    "expanded": expanded, "max_frontier": max_frontier,
                    "visited": len(best_g), "seconds": time.perf_counter() - t0}

        for action, nxt in successors(grid, current):
            ng = g + 1
            if ng < best_g.get(nxt, float("inf")):
                best_g[nxt] = ng
                parents[nxt] = (current, action)
                counter += 1
                heapq.heappush(frontier, (ng + manhattan(nxt, goal), ng, counter, nxt))
        max_frontier = max(max_frontier, len(frontier))

    return {"path": None, "actions": None, "cost": None,
            "expanded": expanded, "max_frontier": max_frontier,
            "visited": len(best_g), "seconds": time.perf_counter() - t0}

## 4. The goal-based agent

The agent ties the pieces together: it holds a **model of the environment**, its **current
state**, an explicit **goal**, the set of **actions**, and a **decision-making component**
(the search). `plan()` produces a sequence of actions; `run()` executes them one at a time and
re-checks each move against the collision test, so execution never blindly trusts the plan.

In [ ]:
class WarehouseAgent:
    """A goal-based agent for the warehouse navigation problem.

    Attributes
    ----------
    grid    : internal model of the environment (what the agent believes the world is like)
    state   : current position (row, col)
    goal    : goal position (row, col)
    plan_   : list of action names still to be executed
    history : positions actually visited during execution
    """

    def __init__(self, grid, start, goal, strategy=bfs):
        self.grid = grid
        self.state = start
        self.start = start
        self.goal = goal
        self.strategy = strategy          # the decision-making component, swappable
        self.plan_ = []
        self.history = [start]
        self.stats = None

    # --- goal test -------------------------------------------------------------
    def at_goal(self):
        """The agent's explicit goal test - the feature that makes it goal-based."""
        return self.state == self.goal

    # --- deliberation ----------------------------------------------------------
    def plan(self):
        """Search the internal model for a collision-free action sequence to the goal."""
        result = self.strategy(self.grid, self.state, self.goal)
        self.stats = result
        self.plan_ = list(result["actions"]) if result["actions"] is not None else None
        return result

    # --- execution -------------------------------------------------------------
    def step(self):
        """Execute the next planned action, re-checking it for collisions first."""
        if not self.plan_:
            return None
        action = self.plan_.pop(0)
        dr, dc = ACTIONS[action]
        nxt = (self.state[0] + dr, self.state[1] + dc)
        if not is_free(self.grid, nxt):                       # defensive re-check
            raise RuntimeError(f"planned action {action} from {self.state} hits an obstacle")
        self.state = nxt
        self.history.append(nxt)
        return action

    def run(self, verbose=False):
        """Plan, then execute the whole plan. Returns True if the goal was reached."""
        result = self.plan()
        if result["path"] is None:
            print("NO PATH: the dispatch area cannot be reached from the loading bay.")
            print(f"         the search exhausted {result['expanded']} reachable states first.")
            return False
        while self.plan_:
            action = self.step()
            if verbose:
                print(f"  {action:<5} -> {self.state}")
        return self.at_goal()

## 5. Solving the warehouse: path found

In [ ]:
def render(grid, path, start, goal):
    """Return the map as a string with the path drawn using '*' (S and G left visible)."""
    out = [row[:] for row in grid]
    for cell in path:
        if cell not in (start, goal):
            out[cell[0]][cell[1]] = "*"
    return "\n".join("".join(row) for row in out)


def summarise(name, result):
    if result["path"] is None:
        print(f"{name}: no path found ({result['expanded']} states expanded)")
        return
    print(f"{name}: path of {result['cost']} moves | states expanded {result['expanded']} | "
          f"max frontier {result['max_frontier']} | {result['seconds'] * 1000:.2f} ms")


agent = WarehouseAgent(grid, start, goal, strategy=bfs)
reached = agent.run()

print("PATH FOUND" if reached else "NO PATH")
print(f"goal reached: {agent.at_goal()}   final position: {agent.state}\n")

res_bfs = agent.stats
print("Action sequence (" + str(res_bfs["cost"]) + " moves):")
print("  " + " -> ".join(res_bfs["actions"]))

print("\nPath as (row, col) coordinates:")
print("  " + " ".join(str(p) for p in res_bfs["path"]))

print("\nWarehouse with the path marked '*':\n")
print(render(grid, res_bfs["path"], start, goal))

### Same problem solved with A*

Both searches return an optimal path of the same length; A* reaches it after expanding far
fewer states, because the heuristic pulls the search towards the goal instead of growing a
circle in every direction.

In [ ]:
res_astar = astar(grid, start, goal)

summarise("BFS", res_bfs)
summarise("A* ", res_astar)
print(f"\nsame cost: {res_bfs['cost'] == res_astar['cost']}  "
      f"(BFS {res_bfs['cost']} moves, A* {res_astar['cost']} moves)")
print(f"states expanded: BFS {res_bfs['expanded']}, A* {res_astar['expanded']} "
      f"({100 * (1 - res_astar['expanded'] / res_bfs['expanded']):.0f}% fewer)")
print("\nA* path drawn on the map:\n")
print(render(grid, res_astar["path"], start, goal))

## 6. Contrast: why a simple reflex agent is not enough

A simple reflex agent chooses its action from the current percept alone. The strongest such
rule available here is "move to whichever neighbouring cell is closest to G".

On *this* map that rule happens to work - which is worth reporting honestly, because it is the
result the experiment gives. But it works by luck of the layout, not by construction: the rule
has no guarantee, no way to accept a temporary increase in distance, and no way to report that
a goal is unreachable. The second map below - an aisle whose only entrance is at the far end -
defeats it in three moves, while the goal-based agent solves both.

In [ ]:
def reflex_agent(grid, start, goal, max_steps=50):
    """Greedy reflex agent: always step to the legal neighbour closest to the goal.

    Returns (trace, outcome). It has no memory of where it has been, so it can loop;
    the loop is detected here only so the demonstration terminates.
    """
    pos, trace, seen = start, [start], {start}
    for _ in range(max_steps):
        if pos == goal:
            return trace, "reached the goal"
        moves = [(manhattan(nxt, goal), action, nxt) for action, nxt in successors(grid, pos)]
        if not moves:
            return trace, f"stuck at {pos}: no legal move"
        _, action, nxt = min(moves)
        pos = nxt
        trace.append(pos)
        if pos in seen:
            return trace, f"looping: returned to {pos} after {len(trace) - 1} moves"
        seen.add(pos)
    return trace, "step limit reached"


trace, outcome = reflex_agent(grid, start, goal)
print("--- reflex agent on the laboratory map ---")
print("trace  :", " ".join(str(p) for p in trace))
print("outcome:", outcome, f"in {len(trace) - 1} moves")
print("Look at the step from (1, 5): both legal moves - Left to (1, 4) and Down to (2, 5) -")
print("increase the Manhattan distance to G from 14 to 15, so the rule is choosing between two")
print("equally bad options and the tie is broken alphabetically ('Down' before 'Left'). Down")
print("happens to be the detour that works; Left would have sent it back to (1, 4) and into an")
print("oscillation. The success is a property of this layout and an arbitrary tie-break, not a")
print("property of the rule.\n")

# An aisle with a single entrance at the far end: the same greedy rule now fails.
AISLE_MAP = """\
#########
#S......#
#######.#
#G......#
#########"""

agrid, astart, agoal = parse_map(AISLE_MAP)
print("--- reflex agent on an aisle with one entrance ---")
print(AISLE_MAP)
atrace, aoutcome = reflex_agent(agrid, astart, agoal)
print("trace  :", " ".join(str(p) for p in atrace))
print("outcome:", aoutcome)

ares = bfs(agrid, astart, agoal)
print(f"\nthe goal-based agent solves the same map in {ares['cost']} moves:")
print("  " + " -> ".join(ares["actions"]))
print()
print(render(agrid, ares["path"], astart, agoal))
print("\nG is two squares below S, so every move out of the corner increases the straight-line")
print("distance. The reflex agent oscillates because it cannot look further ahead than one step;")
print("the goal-based agent accepts the detour because it evaluates whole action sequences.")

## 7. Validating the output

Printing a path is not evidence that the path is correct. Each property that *must* hold is
checked explicitly, so a wrong answer fails loudly rather than looking plausible.

In [ ]:
def validate(grid, start, goal, path, actions):
    """Check every property a valid solution must satisfy. Raises AssertionError otherwise."""
    checks = []

    checks.append(("path starts at S", path[0] == start))
    checks.append(("path ends at G", path[-1] == goal))
    checks.append(("no cell is an obstacle", all(grid[r][c] != OBSTACLE for r, c in path)))
    checks.append(("all cells inside the grid", all(in_bounds(grid, p) for p in path)))

    single_steps = all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1
                       for a, b in zip(path, path[1:]))
    checks.append(("every step moves exactly one square", single_steps))

    consistent = all((b[0] - a[0], b[1] - a[1]) == ACTIONS[act]
                     for a, b, act in zip(path, path[1:], actions))
    checks.append(("actions match the coordinates", consistent))
    checks.append(("action count = path length - 1", len(actions) == len(path) - 1))
    checks.append(("no cell visited twice", len(set(path)) == len(path)))

    for name, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
        assert ok, name
    return True


print("Validating the BFS solution:")
validate(grid, start, goal, res_bfs["path"], res_bfs["actions"])

print("\nValidating the A* solution:")
validate(grid, start, goal, res_astar["path"], res_astar["actions"])

# optimality cross-check: two independent algorithms must agree on the cost
assert res_bfs["cost"] == res_astar["cost"], "BFS and A* disagree on the optimal cost"
print(f"\n  [PASS] BFS and A* agree that the shortest path is {res_bfs['cost']} moves")

# a lower bound the answer cannot beat: the straight-line Manhattan distance
lb = manhattan(start, goal)
assert res_bfs["cost"] >= lb
print(f"  [PASS] cost {res_bfs['cost']} >= Manhattan lower bound {lb} "
      f"(the detour around the shelving costs {res_bfs['cost'] - lb} extra moves)")

## 8. Failure case: an unreachable goal

A path-finder that only ever reports success has not been tested. Here the goal is walled in,
so no path exists and the agent must say so rather than loop, crash, or return a broken path.

In [ ]:
BLOCKED_MAP = """\
#####################
#S....#...........#G#
#.##....##########.##
#....##............##
#.######.###.#.###..#
#........#..........#
#####################"""

bgrid, bstart, bgoal = parse_map(BLOCKED_MAP)
print(BLOCKED_MAP)
print("\nThe dispatch area is sealed off by shelving.\n")

blocked_agent = WarehouseAgent(bgrid, bstart, bgoal, strategy=bfs)
ok = blocked_agent.run()
print(f"\ngoal reached: {ok}")
assert ok is False and blocked_agent.stats["path"] is None
print("[PASS] the agent correctly reports that no path exists")

# A* must reach the same conclusion
assert astar(bgrid, bstart, bgoal)["path"] is None
print("[PASS] A* also reports no path")

# and a start cell that is itself a wall must be rejected at once, not searched
try:
    bfs(grid, (0, 0), goal)
except ValueError as e:
    print(f"[PASS] illegal start rejected: {e}")

## 9. "What if the warehouse were twice as large?"

The Think About It question, answered with measurements instead of intuition. Each cell of the
original map is expanded into a k x k block, so the floor area grows as k^2 while the layout
stays the same. BFS remains correct at every size; the question is what it costs.

In [ ]:
def scale_map(text, k):
    """Blow up each cell of the map into a k x k block (same layout, k^2 times the area)."""
    out = []
    for row in text.strip("\n").split("\n"):
        # widen: an obstacle or free cell becomes k copies of itself; S and G stay single
        # cells (the rest of their block becomes free floor) so the map still parses.
        wide = "".join(ch * k if ch in (OBSTACLE, FREE) else ch + FREE * (k - 1) for ch in row)
        plain = wide.replace(START, FREE).replace(GOAL, FREE)   # the k-1 rows below
        out.append(wide)
        out.extend([plain] * (k - 1))
    return "\n".join(out)


print(f"{'scale k':>8}{'cells':>9}{'free':>7}{'path':>7}"
      f"{'BFS expanded':>15}{'A* expanded':>13}{'BFS ms':>9}{'A* ms':>8}")
for k in (1, 2, 4, 8, 16):
    g, s, gl = parse_map(scale_map(WAREHOUSE_MAP, k))
    rb, ra = bfs(g, s, gl), astar(g, s, gl)
    cells = len(g) * len(g[0])
    free = sum(ch != OBSTACLE for row in g for ch in row)
    print(f"{k:>8}{cells:>9}{free:>7}{rb['cost']:>7}{rb['expanded']:>15}{ra['expanded']:>13}"
          f"{rb['seconds'] * 1000:>9.1f}{ra['seconds'] * 1000:>8.1f}")
    assert rb["cost"] == ra["cost"]

print("\nBFS stays complete and optimal at every size - what grows is the work and the memory:")
print("the visited set and the frontier are both O(free cells), and free cells grow as k^2.")

## 10. Summary block

The figures quoted in `report.pdf`, printed in one place for checking.

In [ ]:
print("=" * 64)
print(f"grid                 : {n_rows} x {n_cols} = {n_rows * n_cols} cells, "
      f"{free_cells} free, {n_rows * n_cols - free_cells} obstacles")
print(f"start / goal         : {start} -> {goal}")
print(f"Manhattan lower bound: {manhattan(start, goal)} moves")
print(f"BFS  : cost {res_bfs['cost']}  expanded {res_bfs['expanded']}  "
      f"max frontier {res_bfs['max_frontier']}  {res_bfs['seconds'] * 1000:.2f} ms")
print(f"A*   : cost {res_astar['cost']}  expanded {res_astar['expanded']}  "
      f"max frontier {res_astar['max_frontier']}  {res_astar['seconds'] * 1000:.2f} ms")
print(f"actions              : {' '.join(a[0] for a in res_bfs['actions'])}")
print(f"unreachable-goal test: path is None -> agent reports failure correctly")
print("=" * 64)